In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules


In [2]:
training_df=pd.read_csv("../Datasets//training_df.csv")
training_df.fillna(0,inplace=True)
training_df.drop(['date','time'],axis=1,inplace=True)
training_df

,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,pressure_msl,...,wind_gusts_10m,soil_temperature_0_to_7cm,soil_temperature_7_to_28cm,soil_temperature_28_to_100cm,soil_temperature_100_to_255cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm,fire
0,10.460000,95.428820,9.7600,9.326071,0.0,0.0,0.0,0.0,0.0,1012.4,...,11.879999,10.760000,13.110001,14.010000,21.910000,0.275,0.318,0.195,0.026,1
1,10.453500,95.428590,9.7535,9.317741,0.0,0.0,0.0,0.0,0.0,1012.4,...,11.879999,10.753500,13.103500,14.003500,21.903500,0.275,0.318,0.195,0.026,1
2,10.414500,95.427210,9.7145,9.267784,0.0,0.0,0.0,0.0,0.0,1012.4,...,11.879999,10.714500,13.064501,13.964500,21.864500,0.275,0.318,0.195,0.026,1
3,9.223500,99.663920,9.1735,8.143780,0.0,0.0,0.0,0.0,0.0,1012.5,...,13.320000,10.423500,12.823500,13.723500,21.923500,0.273,0.324,0.201,0.020,1
4,9.243000,99.663990,9.1930,8.168585,0.0,0.0,0.0,0.0,0.0,1012.5,...,13.320000,10.443000,12.842999,13.743000,21.942999,0.273,0.324,0.201,0.020,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113891,28.476500,26.921050,7.6265,26.616827,0.0,0.0,0.0,0.0,1.0,1015.9,...,38.160000,41.726498,30.776499,26.726500,18.426498,0.042,0.134,0.192,0.235,0
113892,25.435000,36.749325,9.5850,24.854360,0.0,0.0,0.0,0.0,0.0,1009.1,...,17.640000,32.185000,28.734999,23.435000,17.535000,0.135,0.170,0.229,0.293,0
113893,13.914001,82.397710,10.9640,13.423975,0.0,0.0,0.0,0.0,3.0,1012.1,...,18.720000,14.164001,13.364000,12.314000,12.314000,0.231,0.314,0.330,0.342,0
113894,7.604000,52.525845,-1.4960,4.411881,0.0,0.0,0.0,0.0,0.0,1017.0,...,18.720000,10.654000,14.604001,13.604000,7.954000,0.270,0.296,0.347,0.392,0


In [3]:
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113896 entries, 0 to 113895
Data columns (total 31 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   temperature_2m                 113896 non-null  float64
 1   relative_humidity_2m           113896 non-null  float64
 2   dew_point_2m                   113896 non-null  float64
 3   apparent_temperature           113896 non-null  float64
 4   precipitation                  113896 non-null  float64
 5   rain                           113896 non-null  float64
 6   snowfall                       113896 non-null  float64
 7   snow_depth                     113896 non-null  float64
 8   weather_code                   113896 non-null  float64
 9   pressure_msl                   113896 non-null  float64
 10  surface_pressure               113896 non-null  float64
 11  cloud_cover                    113896 non-null  float64
 12  cloud_cover_low               

In [4]:
# Convert numerical data into categorical (binary format for transactions)
def binarize_data(df, threshold=0):
    return df.applymap(lambda x: 1 if x > threshold else 0)

binarized_df = binarize_data(training_df)


C:\Users\abhir\AppData\Local\Temp\ipykernel_9536\2243915037.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: 1 if x > threshold else 0)


In [8]:
# Apply Apriori Algorithm
frequent_itemsets = apriori(binarized_df, min_support=0.01, use_colnames=True)
association_rules_df = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)


c:\Users\abhir\anaconda3\envs\wildfire-prediction\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


MemoryError: Unable to allocate 59.6 GiB for an array with shape (17550, 4, 113896) and data type int64

In [ ]:
# Visualizing associations as a network graph
def plot_network(rules):
    G = nx.DiGraph()
    
    for i, rule in rules.iterrows():
        for antecedent in rule['antecedents']:
            for consequent in rule['consequents']:
                G.add_edge(antecedent, consequent, weight=rule['lift'])
    
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G)
    edges = G.edges(data=True)
    
    nx.draw(G, pos, edge_color='gray', node_color='lightblue', with_labels=True, node_size=2000, font_size=10)
    edge_labels = {(u, v): round(d['weight'], 2) for u, v, d in edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
    
    plt.title("Association Rule Network Graph")
    plt.show()

plot_network(top_lift)
